In [10]:
# FAZ 2: Base Model ve Temel Değerlendirme (Zero-Shot Baseline)
# Bu notebook, eğitilmemiş modelin performansını ölçer

from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import json
from tqdm import tqdm
import sys
import os
import re

# src klasörünü path'e ekle
sys.path.append(os.getcwd())
from src.utils import generate_answer_with_model, parse_model_output

print("Kütüphaneler yüklendi ve src.utils import edildi.")


Kütüphaneler yüklendi ve src.utils import edildi.


In [11]:
# Veri setlerini yükle
train_dataset = load_from_disk("data/train_dataset")
test_dataset = load_from_disk("data/test_dataset")

print(f"Train seti: {len(train_dataset)} örnek")
print(f"Test seti: {len(test_dataset)} örnek")
print(f"\nTest seti özellikleri:")
print(test_dataset.features)


Train seti: 1000 örnek
Test seti: 500 örnek

Test seti özellikleri:
{'bolum': Value('string'), 'konu': Value('string'), 'messages': List({'content': Value('string'), 'role': Value('string')}), 'correct_answer': Value('string'), 'original_answer_index': Value('int64')}


In [12]:
# Base modeli ve tokenizer'ı yükle
model_name = "ytu-ce-cosmos/turkish-gpt2-large-750m-instruct-v0.1"

print(f"Model yükleniyor: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model yüklendi. Cihaz: {next(model.parameters()).device}")
print(f"Tokenizer vocab size: {len(tokenizer)}")


Model yükleniyor: ytu-ce-cosmos/turkish-gpt2-large-750m-instruct-v0.1
Model yüklendi. Cihaz: cuda:0
Tokenizer vocab size: 50258


In [13]:
def parse_model_output(text: str) -> tuple:
    """
    Model çıktısından cevabı parse eder (Basit Format).
    Format:
    Açıklama: ...
    Cevap: ...
    
    Returns:
        (düşünce, cevap, format_uygun): düşünce metni, cevap harfi (A-E), format uygunluğu
    """
    düşünce = None
    cevap = None
    format_uygun = False
    
    # Açıklama kısmını al
    düşünce_pattern = r'Açıklama:(.*?)(?=Cevap:|$)'
    düşünce_match = re.search(düşünce_pattern, text, re.DOTALL | re.IGNORECASE)
    if düşünce_match:
        düşünce = düşünce_match.group(1).strip()
    
    # Cevap kısmını al
    cevap_pattern = r'Cevap:\s*([A-E])'
    cevap_match = re.search(cevap_pattern, text, re.IGNORECASE)
    if cevap_match:
        cevap = cevap_match.group(1).upper()
        format_uygun = True
    else:
        # Fallback: Metinde A-E harflerini ara
        cevap_pattern_fallback = r'\b([A-E])\b'
        cevap_matches = re.findall(cevap_pattern_fallback, text, re.IGNORECASE)
        if cevap_matches:
            cevap = cevap_matches[-1].upper()
    
    return düşünce, cevap, format_uygun

# Test: Parse fonksiyonunu test et
test_outputs = [
    "Açıklama: Bu soruyu çözmek için düşünüyorum...\nCevap: B",
    "Bence cevap C olmalı",
    "Açıklama: Mantıklı...\nCevap: A",
    "Sadece D"
]

print("Parse fonksiyonu test ediliyor:")
for i, test_output in enumerate(test_outputs, 1):
    düşünce, cevap, format_uygun = parse_model_output(test_output)
    print(f"\nTest {i}:")
    print(f"  Input: {test_output[:50]}...")
    print(f"  Düşünce: {düşünce[:30] if düşünce else None}...")
    print(f"  Cevap: {cevap}")
    print(f"  Format Uygun: {format_uygun}")


Parse fonksiyonu test ediliyor:

Test 1:
  Input: Açıklama: Bu soruyu çözmek için düşünüyorum...
Cev...
  Düşünce: Bu soruyu çözmek için düşünüyo...
  Cevap: B
  Format Uygun: True

Test 2:
  Input: Bence cevap C olmalı...
  Düşünce: None...
  Cevap: C
  Format Uygun: False

Test 3:
  Input: Açıklama: Mantıklı...
Cevap: A...
  Düşünce: Mantıklı......
  Cevap: A
  Format Uygun: True

Test 4:
  Input: Sadece D...
  Düşünce: None...
  Cevap: D
  Format Uygun: False


In [14]:
def generate_answer(messages: list, max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    """
    Model'e mesajları gönderip cevap üretir.
    
    Args:
        messages: [{"role": "user", "content": "..."}] (Sadece User)
        max_new_tokens: Maksimum üretilecek token sayısı
        temperature: Generation temperature
    
    Returns:
        Model çıktısı (string)
    """
    # Mesajları formatla (instruct format için)
    # Model System prompt desteklemiyor, veri setinde zaten User içine gömülü
    formatted_prompt = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        # Rolleri basitçe haritala
        if role == "user":
            formatted_prompt += f"### Kullanıcı:\n{content}\n\n"
        elif role == "system": # Eski veri setinden kalma ihtimaline karşı
             formatted_prompt += f"### Sistem:\n{content}\n\n"
            
    formatted_prompt += "### Asistan:\n"
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=1024)
    
    # Cihaza taşı
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Sadece model'in ürettiği kısmı al (prompt'tan sonrası)
    generated_text = full_text[len(formatted_prompt):].strip()
    
    return generated_text

# Test: Bir örnek üzerinde test et
print("Model generation test ediliyor...")
test_example = test_dataset[0]
print(f"Test örneği - Bölüm: {test_example['bolum']}")
# print(f"User: {test_example['messages'][0]['content'][:100]}...")
print(f"Doğru cevap: {test_example['correct_answer']}")

test_response = generate_answer(test_example['messages'], max_new_tokens=256)
print(f"\nModel çıktısı (ilk 200 karakter):\n{test_response[:200]}...")


Model generation test ediliyor...
Test örneği - Bölüm: YGS Denemeleri
Doğru cevap: C

Model çıktısı (ilk 200 karakter):
II. TBMM’nin yeni Türk devletinin temellerinin atılmasında önemli bir rol oynadığı söylenebilir....


In [15]:
# Tüm test setini değerlendir
print("Test seti değerlendiriliyor...")
print(f"Toplam {len(test_dataset)} örnek işlenecek.\n")

results = []

for i, example in enumerate(tqdm(test_dataset, desc="Değerlendirme")):
    # Model çıktısını al
    model_output = generate_answer(example['messages'], max_new_tokens=512)
    
    # Parse et
    düşünce, predicted_answer, format_uygun = parse_model_output(model_output)
    
    # Sonuçları kaydet
    result = {
        'index': i,
        'bolum': example.get('bolum', 'unknown'),
        'konu': example.get('konu', 'unknown'),
        'correct_answer': example['correct_answer'],
        'predicted_answer': predicted_answer,
        'is_correct': (predicted_answer == example['correct_answer']) if predicted_answer else False,
        'format_uygun': format_uygun,
        'düşünce_length': len(düşünce) if düşünce else 0,
        'model_output': model_output  # Tam model çıktısını kaydet
    }
    results.append(result)

print(f"\nDeğerlendirme tamamlandı. {len(results)} örnek işlendi.")


Test seti değerlendiriliyor...
Toplam 500 örnek işlenecek.



Değerlendirme: 100%|██████████| 500/500 [03:58<00:00,  2.10it/s]


Değerlendirme tamamlandı. 500 örnek işlendi.


In [16]:
# Sonuçları analiz et
df_results = pd.DataFrame(results)

# Genel istatistikler
total = len(df_results)
correct = df_results['is_correct'].sum()
format_compliant = df_results['format_uygun'].sum()

accuracy = correct / total * 100
format_accuracy = format_compliant / total * 100

print("="*60)
print("BASELINE DEĞERLENDİRME SONUÇLARI")
print("="*60)
print(f"\n📊 GENEL İSTATİSTİKLER:")
print(f"  Toplam örnek sayısı: {total}")
print(f"  Doğru cevap sayısı: {correct}")
print(f"  Format uygun örnek sayısı: {format_compliant}")

print(f"\n✅ DOĞRULUK METRİKLERİ:")
print(f"  Accuracy: {accuracy:.2f}%")
print(f"  Format Compliance: {format_accuracy:.2f}%")

print(f"\n📝 CEVAP DAĞILIMI:")
print("  Doğru Cevap Dağılımı:")
correct_answer_dist = df_results['correct_answer'].value_counts().sort_index()
for letter, count in correct_answer_dist.items():
    print(f"    {letter}: {count} ({count/total*100:.1f}%)")

print("\n  Tahmin Edilen Cevap Dağılımı:")
predicted_answer_dist = df_results['predicted_answer'].value_counts().sort_index() if df_results['predicted_answer'].notna().any() else pd.Series()
if not predicted_answer_dist.empty:
    for letter, count in predicted_answer_dist.items():
        print(f"    {letter}: {count} ({count/total*100:.1f}%)")
else:
    print("    (Cevap bulunamadı)")

print(f"\n📂 BÖLÜM BAZINDA PERFORMANS:")
bolum_stats = df_results.groupby('bolum').agg({
    'is_correct': ['count', 'sum', 'mean'],
    'format_uygun': 'sum'
}).round(3)

bolum_stats.columns = ['Toplam', 'Doğru', 'Accuracy', 'Format_Uygun']
bolum_stats = bolum_stats.sort_values('Toplam', ascending=False)
print(bolum_stats.head(10))

print(f"\n📏 DÜŞÜNCE UZUNLUK İSTATİSTİKLERİ:")
düşünce_lengths = df_results[df_results['düşünce_length'] > 0]['düşünce_length']
if len(düşünce_lengths) > 0:
    print(f"  Ortalama: {düşünce_lengths.mean():.1f} karakter")
    print(f"  Medyan: {düşünce_lengths.median():.1f} karakter")
    print(f"  Min: {düşünce_lengths.min()} karakter")
    print(f"  Max: {düşünce_lengths.max()} karakter")
    print(f"  Düşünce içeren örnek sayısı: {len(düşünce_lengths)} ({len(düşünce_lengths)/total*100:.1f}%)")
else:
    print("  Hiç düşünce üretilmemiş.")


BASELINE DEĞERLENDİRME SONUÇLARI

📊 GENEL İSTATİSTİKLER:
  Toplam örnek sayısı: 500
  Doğru cevap sayısı: 24
  Format uygun örnek sayısı: 8

✅ DOĞRULUK METRİKLERİ:
  Accuracy: 4.80%
  Format Compliance: 1.60%

📝 CEVAP DAĞILIMI:
  Doğru Cevap Dağılımı:
    A: 88 (17.6%)
    B: 89 (17.8%)
    C: 113 (22.6%)
    D: 98 (19.6%)
    E: 112 (22.4%)

  Tahmin Edilen Cevap Dağılımı:
    A: 15 (3.0%)
    B: 16 (3.2%)
    C: 28 (5.6%)
    D: 16 (3.2%)
    E: 26 (5.2%)

📂 BÖLÜM BAZINDA PERFORMANS:
                            Toplam  Doğru  Accuracy  Format_Uygun
bolum                                                            
YGS Denemeleri                 218      9     0.041             3
KPSS Denemeleri                111      7     0.063             2
TUS                             45      1     0.022             0
Dış Ticaret                     38      0     0.000             1
Felsefe                         17      0     0.000             0
Yönetim Bİlişim Sistemleri      15      2     0

In [17]:
# Sonuçları CSV olarak kaydet
output_file = "baseline_results.csv"

# DataFrame'i kopyala (tam model çıktısını kaydet)
df_export = df_results.copy()
# Not: CSV formatında çok uzun metinler sorun olabilir ama tam çıktıyı koruyoruz
# Eğer CSV açılırken sorun olursa, alternatif olarak JSON formatında da kaydedebiliriz

df_export.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Sonuçlar CSV olarak kaydedildi: {output_file}")
print(f"Toplam model çıktı uzunluğu - Ortalama: {df_export['model_output'].str.len().mean():.1f} karakter")
print(f"Toplam model çıktı uzunluğu - Max: {df_export['model_output'].str.len().max()} karakter")

# Alternatif: JSON formatında da kaydet (tam çıktı için daha güvenli)
results_json_file = "baseline_results.json"
df_export.to_json(results_json_file, orient='records', indent=2, force_ascii=False)
print(f"Sonuçlar JSON olarak da kaydedildi: {results_json_file} (tam çıktı için önerilir)")

# Özet bilgileri de JSON olarak kaydet
summary = {
    'total_samples': int(total),
    'accuracy': float(accuracy),
    'format_compliance': float(format_accuracy),
    'correct_count': int(correct),
    'format_compliant_count': int(format_compliant),
    'correct_answer_distribution': correct_answer_dist.to_dict(),
    'predicted_answer_distribution': predicted_answer_dist.to_dict() if not predicted_answer_dist.empty else {}
}

with open("baseline_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Özet bilgileri kaydedildi: baseline_summary.json")


Sonuçlar CSV olarak kaydedildi: baseline_results.csv
Toplam model çıktı uzunluğu - Ortalama: 224.0 karakter
Toplam model çıktı uzunluğu - Max: 2671 karakter
Sonuçlar JSON olarak da kaydedildi: baseline_results.json (tam çıktı için önerilir)
Özet bilgileri kaydedildi: baseline_summary.json


In [18]:
# Örnek sonuçları göster (doğru ve yanlış örnekler)
print("="*60)
print("ÖRNEK SONUÇLAR")
print("="*60)

print("\n✅ DOĞRU CEVAP ÖRNEKLERİ:")
correct_examples = df_results[df_results['is_correct'] == True].head(3)
for idx, row in correct_examples.iterrows():
    print(f"\nÖrnek {row['index']} (Bölüm: {row['bolum']}):")
    print(f"  Doğru Cevap: {row['correct_answer']}")
    print(f"  Tahmin: {row['predicted_answer']}")
    print(f"  Format Uygun: {row['format_uygun']}")
    if row['düşünce_length'] > 0:
        print(f"  Düşünce Uzunluğu: {row['düşünce_length']} karakter")

print("\n❌ YANLIŞ CEVAP ÖRNEKLERİ:")
wrong_examples = df_results[df_results['is_correct'] == False].head(3)
for idx, row in wrong_examples.iterrows():
    print(f"\nÖrnek {row['index']} (Bölüm: {row['bolum']}):")
    print(f"  Doğru Cevap: {row['correct_answer']}")
    print(f"  Tahmin: {row['predicted_answer']}")
    print(f"  Format Uygun: {row['format_uygun']}")
    if row['düşünce_length'] > 0:
        print(f"  Düşünce Uzunluğu: {row['düşünce_length']} karakter")

print("\n⚠️  FORMAT UYGUN OLMAYAN ÖRNEKLER:")
format_wrong = df_results[df_results['format_uygun'] == False].head(3)
for idx, row in format_wrong.iterrows():
    print(f"\nÖrnek {row['index']} (Bölüm: {row['bolum']}):")
    print(f"  Doğru Cevap: {row['correct_answer']}")
    print(f"  Tahmin: {row['predicted_answer']}")
    print(f"  Model Çıktısı (ilk 150 karakter): {row['model_output'][:150]}...")


ÖRNEK SONUÇLAR

✅ DOĞRU CEVAP ÖRNEKLERİ:

Örnek 7 (Bölüm: İktisat):
  Doğru Cevap: A
  Tahmin: A
  Format Uygun: False

Örnek 17 (Bölüm: YGS Denemeleri):
  Doğru Cevap: E
  Tahmin: E
  Format Uygun: False

Örnek 50 (Bölüm: YGS Denemeleri):
  Doğru Cevap: A
  Tahmin: A
  Format Uygun: False

❌ YANLIŞ CEVAP ÖRNEKLERİ:

Örnek 0 (Bölüm: YGS Denemeleri):
  Doğru Cevap: C
  Tahmin: None
  Format Uygun: False

Örnek 1 (Bölüm: YGS Denemeleri):
  Doğru Cevap: E
  Tahmin: None
  Format Uygun: False

Örnek 2 (Bölüm: Dış Ticaret):
  Doğru Cevap: E
  Tahmin: None
  Format Uygun: False

⚠️  FORMAT UYGUN OLMAYAN ÖRNEKLER:

Örnek 0 (Bölüm: YGS Denemeleri):
  Doğru Cevap: C
  Tahmin: None
  Model Çıktısı (ilk 150 karakter): II. TBMM’nin; I. kurucu olma, II. demokratik olma, III. yenilikçi olma özellikleri vardır....

Örnek 1 (Bölüm: YGS Denemeleri):
  Doğru Cevap: E
  Tahmin: None
  Model Çıktısı (ilk 150 karakter): Bilimsel bilginin bu özelliği, sürekli bir şekilde değişen ve gelişen bir yapıya sahip 